# Export OSNet-x1.0 to ONNX

Exports the OSNet-x1.0 re-ID model (torchreid, Market-1501 weights) to ONNX format.
The resulting file is uploaded to GitHub Releases so any Jetson can download it
via `download_models.py --reid` without needing torch or torchreid installed.

**Steps:**
1. Run all cells in order
2. The last cell downloads `osnet_x1_0_market1501.onnx` to your computer
3. Upload that file to GitHub Releases: `NX-JETSON → Releases → Draft a new release`

In [ ]:
# Cell 1 — Install dependencies
!pip install onnxruntime --quiet


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 101.6 MB/s eta 0:00:0000:0100:01


In [6]:
# Cell 2 — Export OSNet-x1.0 to ONNX
import torch
import torchreid

OUTPUT_FILE = "osnet_x1_0_market1501.onnx"

print("Loading OSNet-x1.0 pretrained weights (Market-1501)...")
model = torchreid.models.build_model("osnet_x1_0", num_classes=1, pretrained=True)
model.eval()

# Input: batch x 3 channels x 256 height x 128 width (RGB, ImageNet-normalized)
# Output: batch x 512 embedding vector
dummy_input = torch.randn(1, 3, 256, 128)

print(f"Exporting to {OUTPUT_FILE}...")
torch.onnx.export(
    model,
    dummy_input,
    OUTPUT_FILE,
    opset_version=11,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
    dynamo=False,  # use legacy TorchScript exporter — avoids onnxscript dependency in PyTorch 2.x
)
print("Export done.")

Loading OSNet-x1.0 pretrained weights (Market-1501)...
Successfully loaded imagenet pretrained weights from "/root/.cache/torch/checkpoints/osnet_x1_0_imagenet.pth"
** The following layers are discarded due to unmatched keys or layer size: ['classifier.weight', 'classifier.bias']
Exporting to osnet_x1_0_market1501.onnx...


/tmp/ipykernel_1195/4094313898.py:16: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(


Export done.


In [7]:
# Cell 3 — Verify: run a dummy inference through the exported ONNX
import onnxruntime as ort
import numpy as np

sess = ort.InferenceSession(OUTPUT_FILE, providers=["CPUExecutionProvider"])
dummy_np = np.random.randn(1, 3, 256, 128).astype(np.float32)
output = sess.run(None, {"input": dummy_np})[0]

assert output.shape == (1, 512), f"Unexpected output shape: {output.shape}"

import os
size_mb = os.path.getsize(OUTPUT_FILE) / 1e6
print(f"Output shape: {output.shape}  ✓")
print(f"File size:    {size_mb:.1f} MB  (expected ~9-10 MB)")
print("Model verified — ready to upload to GitHub Releases.")

ModuleNotFoundError: No module named 'onnxruntime'

In [ ]:
# Cell 4 — Copy to Google Drive
# The file will appear in the root of your Drive as osnet_x1_0_market1501.onnx
from google.colab import drive
import shutil, os

drive.mount("/content/drive")

src = OUTPUT_FILE
dst = "/content/drive/MyDrive/osnet_x1_0_market1501.onnx"
shutil.copy(src, dst)
print(f"Copied to Google Drive: {dst} ({os.path.getsize(dst)/1e6:.1f} MB)")